In [0]:
%pip install FlagEmbedding

In [0]:
import re
import pandas as pd
from pyspark.sql import functions as F, types as T

# ---- Layer ----
CATALOG = "us_gmsgq_dev"
ALYT    = "gms_us_alyt"        # analytics layer — you have read+write here
MART    = "gms_us_mart"        # read-only for you (source data lives here)

# ---- RAW reference inputs (your uploaded tables) ----
RAW_CLINICAL = f"{CATALOG}.{ALYT}.Clinicalid_deviations"
RAW_DOCS     = f"{CATALOG}.{ALYT}.documents_number_deviations"
RAW_ACRONYM  = f"{CATALOG}.{ALYT}.Acronyms_other_deviations"
RAW_CRO      = f"{CATALOG}.{ALYT}.Cro_deviations"

# ---- SOURCE deviation data (read-only, in mart) ----
SOURCE_TABLE = f"{CATALOG}.{MART}.tw_deviation_data_formatted_rdq"

# ---- OUTPUT lookups (MUST be in analytics layer — you can't write to mart) ----
REF_CLINICAL = f"{CATALOG}.{ALYT}.ref_clinical_norm"
REF_DOCS     = f"{CATALOG}.{ALYT}.ref_docs_norm"
REF_ACRONYM  = f"{CATALOG}.{ALYT}.ref_acronym_norm"
REF_CRO      = f"{CATALOG}.{ALYT}.ref_cro_norm"
REF_UNIFIED  = f"{CATALOG}.{ALYT}.ref_glossary_unified"

# ---- The contract every lookup MUST expose (extra cols allowed) ----
LOOKUP_SCHEMA = ["key_norm", "entity_type", "canonical_id", "enrichment_text"]

# Fuzzy-match acceptance threshold for CRO/system names (0-100 rapidfuzz scale)
CRO_FUZZY_THRESHOLD = 88

print("Config loaded.")
print("  Inputs :", RAW_CLINICAL, RAW_DOCS, RAW_ACRONYM, RAW_CRO, sep="\n           ")
print("  Source :", SOURCE_TABLE)
print("  Output :", REF_UNIFIED)

In [0]:
import re
import pandas as pd
from pyspark.sql import functions as F, types as T

# ============================================================================
# SECTION A — CLINICAL IDs
# ============================================================================
CLINICAL_ID_ALTERNATIVES = [
    r"TAK[-\s_]?\d{2,4}[-_/]\d{3,4}",
    r"TAK[-\s_]?\d{2,4}",
    r"MLN[-\s_]?\d{3,4}[-_/]CCT-\d{2,4}",
    r"MLN[-\s_]?\d{3,4}[-_/]\d{2,4}",
    r"MLN[-\s_]?\d{3,4}",
    r"SHP[-\s_]?\d{3,4}[-_/]\d{2,4}",
    r"SHP[-\s_]?\d{3,4}",
    r"HGT[-\s_]?[A-Z]{2,4}[-_/]\d{2,4}",
    r"CCT[-_]?\d{2,4}",
    r"DEN[-_]?\d{2,4}",
    r"C\d{5}",
]
ID_REGEX = re.compile("|".join(CLINICAL_ID_ALTERNATIVES), flags=re.IGNORECASE)

def _which_prefix(u):
    for p in ("TAK", "MLN", "SHP", "HGT", "CCT", "DEN"):
        if u.startswith(p):
            return p
    if re.match(r"C\d{5}$", u):
        return "INTERNAL"
    return "UNKNOWN"

def parse_clinical_id(raw):
    u = re.sub(r"-{2,}", "-", re.sub(r"[\s_/]+", "-", str(raw).upper())).strip("-")
    prefix = _which_prefix(u)
    compound = suffix = None
    if prefix in ("TAK", "MLN", "SHP"):
        m = re.match(prefix + r"-?(\d+)(?:-(.+))?$", u)
        if m: compound, suffix = m.group(1), m.group(2)
        key = (prefix + "-" + compound) if compound else u
    elif prefix == "HGT":
        m = re.match(r"HGT-([A-Z]{2,4})-(\d+)$", u)
        if m: compound, suffix = m.group(1), m.group(2)
        key = ("HGT-" + compound) if compound else u
    elif prefix in ("CCT", "DEN"):
        m = re.match(prefix + r"-?(\d+)$", u)
        compound, suffix, key = prefix, (m.group(1) if m else None), u
    elif prefix == "INTERNAL":
        m = re.match(r"C(\d+)$", u)
        compound, key = (m.group(1) if m else None), u
    else:
        key = u
    return {"raw": raw, "canonical": u, "prefix": prefix,
            "compound_number": compound, "study_suffix": suffix, "normalized_key": key}

@F.udf(T.ArrayType(T.StringType()))
def extract_clinical_keys_udf(text):
    if not text: return []
    keys = set()
    for m in ID_REGEX.finditer(str(text)):
        p = parse_clinical_id(m.group(0))
        if p["normalized_key"]: keys.add(p["normalized_key"])
        if p["canonical"]:      keys.add(p["canonical"])
    return list(keys)

@F.udf(T.ArrayType(T.StringType()))
def clinical_keys_from_ref_udf(protocol, dev_name):
    keys = set()
    for src in (protocol, dev_name):
        if src and str(src).strip():
            p = parse_clinical_id(str(src))
            if p["normalized_key"]: keys.add(p["normalized_key"])
            if p["canonical"]:      keys.add(p["canonical"])
    return list(keys)

# ============================================================================
# SECTION B — DOCUMENTS  (mirrors your dev_01_doc SQL patterns)
# ============================================================================
DOC_PATTERNS = [
    r"\bSOP-\d+", r"\bSPEC-\d+", r"\bMTHD-\d+", r"\bMTD-\d+",
    r"\bPROC-\d+", r"\bTOOL-\d+", r"\bFORM-\d+", r"\bWI-\d+",
]
DOC_REGEX = re.compile("|".join(DOC_PATTERNS), flags=re.IGNORECASE)
DOC_PREFIXES = r"(SOP|SPEC|MTHD|MTD|PROC|TOOL|FORM|WI)"

def norm_doc(v):
    if not v: return None
    u = re.sub(r"-{2,}", "-", re.sub(r"[\s_]+", "-", str(v).upper())).strip("-")
    if not re.match(DOC_PREFIXES + r"-?\d", u): return None
    u = re.sub(r"^MTD-", "MTHD-", u)          # unify legacy method prefix
    return u

norm_doc_udf = F.udf(norm_doc, T.StringType())

@F.udf(T.ArrayType(T.StringType()))
def extract_doc_keys_udf(text):
    if not text: return []
    out = set()
    for m in DOC_REGEX.finditer(str(text)):
        k = norm_doc(m.group(0))
        if k: out.add(k)
    return list(out)

@F.udf(T.ArrayType(T.StringType()))
def legacy_doc_keys_udf(v):
    if not v: return []
    out = set()
    for chunk in str(v).split(";;"):
        chunk = chunk.strip()
        if not chunk or chunk.lower() == "n/a": continue
        for m in re.finditer(DOC_PREFIXES + r"[-\s]?\d{3,7}", chunk, re.I):
            k = norm_doc(m.group(0))
            if k: out.add(k)
    return list(out)

# ============================================================================
# SECTION C — ACRONYMS  (regex candidates + KNOWN-SET filter, SYMMETRIC norm)
# ============================================================================
ACRONYM_CANDIDATE = re.compile(
    r"\(?[A-Z]{2,}\)?(?:[-/][A-Z0-9]+)?|\([A-Z]{2,}\)\s?[A-Z]{2,}")

def norm_acr(v):
    """Primary acronym key: trim, collapse spaces, uppercase."""
    if not v: return None
    u = re.sub(r"\s+", " ", str(v).strip()).upper()
    return u or None

def strip_acr(v):
    """Punctuation-stripped variant: '(EU) CTR' -> 'EU CTR'. Symmetric helper."""
    u = norm_acr(v)
    if not u: return None
    s = re.sub(r"[^A-Z0-9 ]", " ", u)
    s = re.sub(r"\s+", " ", s).strip()
    return s or None

norm_acr_udf  = F.udf(norm_acr,  T.StringType())
strip_acr_udf = F.udf(strip_acr, T.StringType())

def make_extract_acronym_udf(known_keys):
    """Factory: extracts acronyms present in the known set.
       Tries BOTH the norm_acr form AND the stripped form for symmetry.
       Accepts a plain Python set (Spark Connect compatible — no broadcast needed)."""
    @F.udf(T.ArrayType(T.StringType()))
    def _udf(text):
        if not text: return []
        known = known_keys
        out = set()
        for m in ACRONYM_CANDIDATE.finditer(str(text)):
            raw = m.group(0)
            k1 = norm_acr(raw)
            k2 = strip_acr(raw)
            if k1 and k1 in known:
                out.add(k1)
            elif k2 and k2 in known:      # fall back to punctuation-stripped match
                out.add(k2)
        return list(out)
    return _udf

# ============================================================================
# SECTION D — CRO / SYSTEMS  (normalize only; matched via fuzzy at join time)
# ============================================================================
def norm_name(v):
    if not v: return None
    u = re.sub(r"[^A-Z0-9 ]", " ", str(v).upper())
    u = re.sub(r"\s+", " ", u).strip()
    return u or None

norm_name_udf = F.udf(norm_name, T.StringType())

@F.udf(T.ArrayType(T.StringType()))
def cro_variants_udf(name, disp, alias):
    return list({norm_name(x) for x in (name, disp, alias) if x and norm_name(x)})

print("Shared UDFs registered: clinical, doc, acronym(factory, symmetric), cro.")

In [0]:
ac = spark.table(RAW_ACRONYM)

# Guard: don't let short acronyms shadow real clinical/doc IDs
ID_LIKE = r"^(TAK|MLN|SHP|HGT|CCT|DEN|SOP|SPEC|MTHD|MTD|FORM|TOOL|WI|PROC)[-\s]?\d"

ref_acronym = (
    ac.withColumn("key_norm", norm_acr_udf(F.col("Acronym")))
      .filter(F.col("key_norm").isNotNull())
      .filter(F.length("key_norm") >= 2)                      # drop 1-char noise
      .filter(~F.col("key_norm").rlike(ID_LIKE))              # don't shadow IDs
      .filter(F.col("Category").isin("Medical", "Industry"))  # keep relevant senses
      .groupBy("key_norm")
      .agg(
          F.concat_ws(" | ", F.collect_set("Definition")).alias("senses"),
          F.first("Acronym", ignorenulls=True).alias("canonical_id"),
      )
      .withColumn("entity_type", F.lit("ACRONYM"))
      .withColumn("enrichment_text", F.concat(F.lit("Acronym expansions: "), F.col("senses")))
      .select(*LOOKUP_SCHEMA)
)

(ref_acronym.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(REF_ACRONYM))
spark.sql(f"COMMENT ON TABLE {REF_ACRONYM} IS "
          "'Sense-aware acronym lookup: key_norm->all definitions. Built from Acronyms_other_deviations.'")

display(spark.table(REF_ACRONYM).limit(20))

# ----------------------------------------------------------------------------
# Build the acronym known-set (BOTH norm + stripped variants) for extraction.
# Symmetric with make_extract_acronym_udf, which also tries both forms.
# ----------------------------------------------------------------------------
base_keys = {
    r["key_norm"]
    for r in spark.table(REF_ACRONYM).select("key_norm").distinct().collect()
    if r["key_norm"]
}
# add punctuation-stripped variants so 'EU CTR' resolves to '(EU) CTR'
acr_key_set = base_keys | {strip_acr(k) for k in base_keys if strip_acr(k)}

assert acr_key_set, "Acronym known-set is empty — check REF_ACRONYM built correctly"

# spark.sparkContext.broadcast() is not available on Spark Connect (Serverless).
# Pass the plain Python set directly; UDF closure captures it at definition time.
extract_acronym_keys_udf = make_extract_acronym_udf(acr_key_set)
print(f"Acronym known-set size: {len(acr_key_set):,} "
      f"(base {len(base_keys):,} + stripped variants)")

# ----------------------------------------------------------------------------
# Build the acronym-expansion map for TA/modality expansion used in Cell 4.
# ----------------------------------------------------------------------------
_acr_pd = (ac.filter(F.col("Category").isin("Medical", "Industry"))
             .select("Acronym", "Definition").toPandas())
ACR_MAP = {norm_acr(a): d for a, d in zip(_acr_pd["Acronym"], _acr_pd["Definition"]) if norm_acr(a)}

TA_OVERRIDES = {
    "NS": "Neuroscience", "ONC": "Oncology", "GI": "Gastroenterology",
    "RARE": "Rare Diseases", "PDT": "Plasma-Derived Therapies",
}
def expand_abbr(v):
    if v is None or str(v).strip() == "":
        return ""
    u = norm_acr(v)
    return TA_OVERRIDES.get(u) or ACR_MAP.get(u) or v
expand_abbr_udf = F.udf(expand_abbr, T.StringType())

print(f"Acronym expansion map size: {len(ACR_MAP):,}")

In [0]:
clin = spark.table(RAW_CLINICAL)

# Raw columns have spaces/parens → wrap in backticks. Adjust if CSV upload renamed them.
clin_enriched = clin.withColumn(
    "enrichment_text",
    F.concat_ws("\n",
        F.concat(F.lit("Generic: "),   F.coalesce(F.col("Generic Name"), F.lit(""))),
        F.concat(F.lit("Brand: "),     F.coalesce(F.col("Brand Name"), F.lit(""))),
        F.concat(F.lit("Modality: "),  expand_abbr_udf(F.col("Modality")),
                 F.lit(" "),           F.coalesce(F.col("Modality_Detail"), F.lit(""))),
        F.concat(F.lit("Mechanism: "), F.coalesce(F.col("Mechanism"), F.lit(""))),
        F.concat(F.lit("Target: "),    F.coalesce(F.col("Target Long Name"), F.lit(""))),
        F.concat(F.lit("Therapeutic Area: "), expand_abbr_udf(F.col("PF_TherapeuticArea"))),
        F.concat(F.lit("Indication: "), F.coalesce(F.col("IND_DESC"), F.lit(""))),
        F.concat(F.lit("Phase: "),     F.coalesce(F.col("`Phase (Nominal)`"), F.lit(""))),
    ),
)

ref_clinical = (
    clin_enriched
    .withColumn("key_norm", F.explode(
        clinical_keys_from_ref_udf(F.col("`Protocol Number`"), F.col("Development_Name"))))
    .filter(F.col("key_norm").isNotNull() & (F.col("key_norm") != ""))
    .withColumn("entity_type", F.lit("CLINICAL_ID"))
    .withColumn("canonical_id", F.coalesce(F.col("Development_Name"), F.col("`Protocol Number`")))
    .select(*LOOKUP_SCHEMA)
    .dropDuplicates(["key_norm"])
)

(ref_clinical.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(REF_CLINICAL))
spark.sql(f"COMMENT ON TABLE {REF_CLINICAL} IS "
          "'Clinical-ID lookup (compound+study keys); TA/modality expanded via acronym map. From Clinicalid_deviations.'")

In [0]:
docs = spark.table(RAW_DOCS)

base = docs.select(
    F.col("document_number__v").alias("canonical_id"),
    F.col("title__v").alias("title"),
    "previous_document_number__c",
    "legacy_numbers__c",
)

primary  = base.select("canonical_id", "title",
                       norm_doc_udf(F.col("canonical_id")).alias("key_norm"))
previous = base.select("canonical_id", "title",
                       norm_doc_udf(F.col("previous_document_number__c")).alias("key_norm"))
legacy   = base.select("canonical_id", "title",
                       F.explode(legacy_doc_keys_udf(F.col("legacy_numbers__c"))).alias("key_norm"))

ref_docs = (
    primary.unionByName(previous).unionByName(legacy)
    .filter(F.col("key_norm").isNotNull())
    .withColumn("entity_type", F.lit("DOC"))
    .withColumn("enrichment_text",
                F.concat(F.lit("Document Title: "), F.coalesce(F.col("title"), F.lit(""))))
    .select(*LOOKUP_SCHEMA)
    .dropDuplicates(["key_norm"])     # collisions surfaced in Cell 8
)

(ref_docs.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(REF_DOCS))
spark.sql(f"COMMENT ON TABLE {REF_DOCS} IS "
          "'Controlled-doc lookup; current+previous+legacy numbers all key to title__v. From Documents_numbers_deviations.'")

display(spark.table(REF_DOCS).limit(20))

In [0]:
cro = spark.table(RAW_CRO).select(
    "id", "type", "name", "displayName", "alias", "description", "businessCriticality")

ref_cro = (
    cro.withColumn("variant", F.explode(cro_variants_udf("name", "displayName", "alias")))
       .withColumn("key_norm", F.col("variant"))
       .filter(F.col("key_norm").isNotNull() & (F.length("key_norm") >= 3))
       .withColumn("entity_type", F.lit("SYSTEM_CRO"))
       .withColumn("canonical_id", F.col("name"))
       .withColumn("enrichment_text", F.concat(
           F.lit("System/Vendor: "), F.coalesce(F.col("name"), F.lit("")),
           F.lit(" — "), F.coalesce(F.col("description"), F.lit("")))) 
       .withColumn("match_mode", F.lit("FUZZY"))
       .select(*LOOKUP_SCHEMA, "match_mode")
       .dropDuplicates(["key_norm"])
)

(ref_cro.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(REF_CRO))
spark.sql(f"COMMENT ON TABLE {REF_CRO} IS "
          "'CRO/application lookup (name+displayName+alias variants). match_mode=FUZZY — resolve via rapidfuzz. From Cro_deviations.'")

display(spark.table(REF_CRO).limit(20))

In [0]:
exact_sources = (
    spark.table(REF_CLINICAL).select(*LOOKUP_SCHEMA)
    .unionByName(spark.table(REF_DOCS).select(*LOOKUP_SCHEMA))
    .unionByName(spark.table(REF_ACRONYM).select(*LOOKUP_SCHEMA))
    .withColumn("match_mode", F.lit("EXACT"))
)
cro_source = spark.table(REF_CRO).select(*LOOKUP_SCHEMA, "match_mode")

unified = exact_sources.unionByName(cro_source)

(unified.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(REF_UNIFIED))
spark.sql(f"COMMENT ON TABLE {REF_UNIFIED} IS "
          "'Unified glossary: key_norm->enrichment_text across CLINICAL_ID/DOC/ACRONYM (EXACT) + SYSTEM_CRO (FUZZY).'")

print("Unified glossary counts by source:")
display(spark.table(REF_UNIFIED).groupBy("entity_type", "match_mode").count().orderBy("entity_type"))

In [0]:
# ---- (a) coverage + collisions per source ----
print("=== Per-source key coverage & collisions ===")
for name, tbl in [("clinical", REF_CLINICAL), ("docs", REF_DOCS),
                  ("acronym", REF_ACRONYM), ("cro", REF_CRO)]:
    df = spark.table(tbl)
    total = df.count()
    collisions = (df.groupBy("key_norm")
                    .agg(F.countDistinct("canonical_id").alias("n"))
                    .filter("n > 1"))
    n_coll = collisions.count()
    flag = "  ⚠️ BUG (expected only for acronym)" if (n_coll and name in ("clinical", "docs")) else ""
    print(f"{name:9s}: {total:6,} keys | {n_coll:4,} colliding key_norm{flag}")
    if n_coll:
        display(collisions.join(df, "key_norm")
                .select("key_norm", "canonical_id").orderBy("key_norm").limit(20))

# ---- (b) reach: % of events hitting >=1 key per entity type ----
EVENT_EXTRACTED = f"{CATALOG}.{ALYT}.tw_deviation_event_extracted"   # from Step A
try:
    events = spark.table(EVENT_EXTRACTED)
    total_events = events.count()
    g = spark.table(REF_UNIFIED).select("key_norm", "entity_type").distinct()

    hit = (events.select("Event_Number", F.explode("extracted_keys").alias("key_norm"))
                 .join(g, "key_norm")
                 .groupBy("entity_type")
                 .agg(F.countDistinct("Event_Number").alias("events_hit")))

    print(f"\n=== Reach across {total_events:,} events ===")
    display(hit.withColumn("reach_pct",
                           F.round(F.col("events_hit") / total_events * 100, 1))
              .orderBy(F.desc("events_hit")))

    # overall: any entity resolved
    any_hit = (events.select("Event_Number", F.explode("extracted_keys").alias("key_norm"))
                     .join(g, "key_norm").select("Event_Number").distinct().count())
    print(f"Events resolving ANY glossary entity: {any_hit:,} "
          f"({any_hit/total_events*100:.1f}%)   ← your per-source 40.4%-style number")
except Exception as e:
    print(f"\n(reach skipped — run Step A first to build {EVENT_EXTRACTED}: {e})")

In [0]:
# ============================================================================
# SECTION I — BGE-M3 EMBEDDINGS TABLE  (three tiers)
#
# Three text representations → three 1024-dim embedding columns:
#   embedding      (fine)  — full enrichment_text: rich context, best for
#                            semantic paragraph-level retrieval
#   mid_embedding  (mid)   — entity_type + canonical_id: structured label,
#                            good for identity/concept matching
#   core_embedding (core)  — canonical_id alone: pure name/ID matching
#
# One BGE-M3 encode pass over all three text lists concatenated → split.
# Three Delta Sync VS indexes created in Cell 11 (one per vector column).
# ============================================================================
import time
from pyspark.sql import functions as F, types as T
from FlagEmbedding import BGEM3FlagModel

EMB_TABLE       = f"{CATALOG}.{ALYT}.deviation_embeddings"
EMB_INDEX_FINE  = f"{CATALOG}.{ALYT}.deviation_emb_fine_idx"
EMB_INDEX_MID   = f"{CATALOG}.{ALYT}.deviation_emb_mid_idx"
EMB_INDEX_CORE  = f"{CATALOG}.{ALYT}.deviation_emb_core_idx"
EMB_DIM         = 1024
VS_ENDPOINT     = "deviation-retrieval-vs"

# ── 1. Load BGE-M3 on GPU driver (FP16 for H100) ──────────────────────────
print("Loading BAAI/bge-m3 (first run downloads ~3 GB from HuggingFace) …")
bge = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
print("Model loaded.")

# ── 2. Pull unified glossary to driver + derive three text tiers ───────────
src_pd = (
    spark.table(REF_UNIFIED)
    .select(
        F.concat_ws("__", F.col("entity_type"), F.col("key_norm")).alias("emb_id"),
        "key_norm", "entity_type", "canonical_id", "enrichment_text",
    )
    .toPandas()
)

src_pd["embedding_text"]      = src_pd["enrichment_text"].fillna("")
src_pd["mid_embedding_text"]  = (
    src_pd["entity_type"].fillna("") + ": " + src_pd["canonical_id"].fillna("")
).str.strip()
src_pd["core_embedding_text"] = src_pd["canonical_id"].fillna("")

n = len(src_pd)
print(f"Rows: {n:,}  |  Total encode calls: {n * 3:,} (3 tiers in one pass)")

# ── 3. Single encode pass — all three text lists concatenated ─────────────
# Concatenate [fine | mid | core], encode once, then split by n.
all_texts = (
    src_pd["embedding_text"].tolist()
    + src_pd["mid_embedding_text"].tolist()
    + src_pd["core_embedding_text"].tolist()
)

BATCH = 256
all_vecs = []
t0 = time.time()
for i in range(0, len(all_texts), BATCH):
    chunk = all_texts[i : i + BATCH]
    out = bge.encode(
        chunk,
        batch_size=len(chunk),
        max_length=512,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False,
    )
    all_vecs.extend(out["dense_vecs"].tolist())

print(f"Encoded {len(all_vecs):,} vectors in {time.time()-t0:.1f}s  |  dim={len(all_vecs[0])}")
assert len(all_vecs[0]) == EMB_DIM

# Split back into three tiers
src_pd["embedding"]      = all_vecs[:n]
src_pd["mid_embedding"]  = all_vecs[n : 2 * n]
src_pd["core_embedding"] = all_vecs[2 * n :]

# ── 4. Write Delta table with CDF + PRIMARY KEY ────────────────────────────
emb_schema = T.StructType([
    T.StructField("emb_id",             T.StringType(),             False),
    T.StructField("key_norm",            T.StringType(),             True),
    T.StructField("entity_type",         T.StringType(),             True),
    T.StructField("canonical_id",        T.StringType(),             True),
    T.StructField("enrichment_text",     T.StringType(),             True),
    T.StructField("embedding_text",      T.StringType(),             True),
    T.StructField("mid_embedding_text",  T.StringType(),             True),
    T.StructField("core_embedding_text", T.StringType(),             True),
    T.StructField("embedding",           T.ArrayType(T.FloatType()), True),
    T.StructField("mid_embedding",       T.ArrayType(T.FloatType()), True),
    T.StructField("core_embedding",      T.ArrayType(T.FloatType()), True),
])

emb_df = spark.createDataFrame(src_pd, schema=emb_schema)

(emb_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(EMB_TABLE))

spark.sql(f"""
    ALTER TABLE {EMB_TABLE}
    SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")

try:
    spark.sql(f"ALTER TABLE {EMB_TABLE} ADD CONSTRAINT pk_emb_id PRIMARY KEY (emb_id)")
    print("PRIMARY KEY constraint declared.")
except Exception as e:
    if "already exists" in str(e).lower():
        print("PRIMARY KEY constraint already exists — skipping.")
    else:
        raise

row_count = spark.table(EMB_TABLE).count()
print(f"\n✓ {EMB_TABLE}")
print(f"  Rows: {row_count:,}  |  dim: {EMB_DIM}  |  CDF: on  |  PK: emb_id")
print(f"  Columns: embedding (fine) | mid_embedding | core_embedding")

In [0]:
# ============================================================================
# SECTION J — VECTOR SEARCH ENDPOINT + THREE DELTA SYNC INDEXES
#
# One endpoint hosts all three indexes (same source table, different vector
# column each). VS requires one index per vector column for Delta Sync.
#
# Prerequisites:
#   1. Cell 10 completed: EMB_TABLE has CDF + UC PRIMARY KEY on emb_id
#   2. Your user has CAN_USE on the VS endpoint (Workspace UI:
#      Machine Learning > Vector Search > <endpoint> > Permissions)
# ============================================================================
import time
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# ── 1. Create endpoint ────────────────────────────────────────────────────
try:
    w.vector_search_endpoints.create_endpoint(
        name=VS_ENDPOINT,
        endpoint_type="STORAGE_OPTIMIZED",
    )
    print(f"Creating endpoint '{VS_ENDPOINT}' … (takes ~5–10 min first time)")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Endpoint '{VS_ENDPOINT}' already exists.")
    else:
        raise

for attempt in range(40):
    ep = w.vector_search_endpoints.get_endpoint(VS_ENDPOINT)
    state = str(ep.endpoint_status.state)
    print(f"  [{attempt+1:02d}] Endpoint: {state}")
    if "ONLINE" in state:
        break
    time.sleep(30)
else:
    raise TimeoutError(f"Endpoint '{VS_ENDPOINT}' did not reach ONLINE.")

# ── 2. Helper: create-or-sync one index ───────────────────────────────────
META_COLS = ["emb_id", "key_norm", "entity_type", "canonical_id", "enrichment_text"]

def _ensure_index(index_name: str, vector_col: str) -> None:
    """Create the Delta Sync index if new; sync it if it already exists."""
    try:
        w.vector_search_indexes.create_index(
            name=index_name,
            endpoint_name=VS_ENDPOINT,
            primary_key="emb_id",
            index_type="DELTA_SYNC",
            delta_sync_index_spec={
                "source_table": EMB_TABLE,
                "embedding_vector_columns": [
                    {"name": vector_col, "embedding_dimension": EMB_DIM}
                ],
                "pipeline_type": "TRIGGERED",
                "columns_to_sync": META_COLS,
            },
        )
        print(f"  Created : {index_name}  ({vector_col})")
    except Exception as e:
        if "already exists" in str(e).lower():
            w.vector_search_indexes.sync_index(index_name)
            print(f"  Synced  : {index_name}  ({vector_col})")
        else:
            raise

    for attempt in range(40):
        idx = w.vector_search_indexes.get_index(index_name)
        state = str(idx.status.detailed_state)
        print(f"    [{attempt+1:02d}] {state}")
        if "ONLINE" in state:
            return
        time.sleep(30)
    raise TimeoutError(f"Index '{index_name}' did not reach ONLINE.")

# ── 3. Create / sync all three indexes ────────────────────────────────────
print("\n── Fine index (embedding — full enrichment_text) ──")
_ensure_index(EMB_INDEX_FINE, "embedding")

print("\n── Mid index (mid_embedding — entity_type + canonical_id) ──")
_ensure_index(EMB_INDEX_MID, "mid_embedding")

print("\n── Core index (core_embedding — canonical_id only) ──")
_ensure_index(EMB_INDEX_CORE, "core_embedding")

print(f"\n✓ Vector Search stack ready")
print(f"  Endpoint   : {VS_ENDPOINT}")
print(f"  Fine  idx  : {EMB_INDEX_FINE}")
print(f"  Mid   idx  : {EMB_INDEX_MID}")
print(f"  Core  idx  : {EMB_INDEX_CORE}")
print("\nRe-sync after re-embedding: call _ensure_index(...) or sync_index() per index.")

In [0]:
# ============================================================================
# SECTION K — SIMILARITY SEARCH  (per-tier)
#
# Use the FINE index when your query is a full sentence/paragraph.
# Use the MID  index when you want concept/label-level matching.
# Use the CORE index when you know roughly what the ID looks like.
#
# Rule: embed the query with the SAME BGE-M3 params used at index time.
# ============================================================================
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

QUERY       = "TAK-279 randomization protocol deviation"
NUM_RESULTS = 5

def _embed(text: str) -> list[float]:
    """Embed a single query string with BGE-M3 (same params as indexing)."""
    out = bge.encode(
        [text],
        batch_size=1,
        max_length=512,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False,
    )
    return out["dense_vecs"][0].tolist()

def _search(index_name: str, query_vec: list[float], label: str) -> None:
    results = w.vector_search_indexes.query_index(
        index_name=index_name,
        columns=["emb_id", "entity_type", "key_norm", "canonical_id", "enrichment_text"],
        query_vector=query_vec,
        num_results=NUM_RESULTS,
    )
    print(f"── {label} ──  query: '{QUERY}'")
    for row in results.result.data_array:
        emb_id, entity_type, key_norm, canonical_id, enrichment_text, score = row
        print(f"  {score:.4f}  [{entity_type}]  {key_norm}  →  {canonical_id}")
        if enrichment_text:
            print(f"           {enrichment_text[:120]}")
    print()

q_vec = _embed(QUERY)

_search(EMB_INDEX_FINE, q_vec, "FINE  (full enrichment_text)")
_search(EMB_INDEX_MID,  q_vec, "MID   (entity_type + canonical_id)")
_search(EMB_INDEX_CORE, q_vec, "CORE  (canonical_id only)")

In [0]:
# ============================================================================
# DIAGNOSTICS
# ============================================================================
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# ── 1. Check embedding table ──────────────────────────────────────────────
try:
    df = spark.table(EMB_TABLE)
    row_count = df.count()
    sample = df.select(
        F.size(F.col("embedding")).alias("fine_dim"),
        F.size(F.col("mid_embedding")).alias("mid_dim"),
        F.size(F.col("core_embedding")).alias("core_dim"),
    ).first()
    print(f"✓ {EMB_TABLE}")
    print(f"  Rows           : {row_count:,}")
    print(f"  fine_dim       : {sample['fine_dim']}")
    print(f"  mid_dim        : {sample['mid_dim']}")
    print(f"  core_dim       : {sample['core_dim']}")
except Exception as e:
    print(f"✗ Embedding table missing or empty: {e}")

# ── 2. Check VS index status ──────────────────────────────────────────────
print()
for label, idx_name in [("FINE", EMB_INDEX_FINE), ("MID", EMB_INDEX_MID), ("CORE", EMB_INDEX_CORE)]:
    try:
        idx = w.vector_search_indexes.get_index(idx_name)
        state = str(idx.status.detailed_state)
        count = getattr(idx.status, 'indexed_row_count', 'n/a')
        print(f"  [{label}]  {state}  |  indexed rows: {count}")
    except Exception as e:
        print(f"  [{label}]  NOT FOUND — {e}")